# English → Telugu dubbing, in the speaker's own voiceEnd-to-end. Upload an English recording, get back Telugu audio that keeps theoriginal background, lands on the original timings, and is spoken in a voicecloned from the recording itself.**Runtime: Colab GPU (T4 is enough).** Runtime → Change runtime type → T4 GPU.| Step | What it does | Why it matters ||---|---|---|| 1 | Setup | || 2 | Upload | || 3 | **Separate voice from background** | Your audio has music/ambience. We keep it and dub only the voice. || 4 | Transcribe (Whisper large-v3) | Word-level timings become the dubbing slots || 5 | **Translate like a dubbing translator** | Meaning + emotion + a duration budget, not literal MT || 6 | Pick a reference clip | Zero-shot cloning copies whatever is in this clip || 7 | **Speak Telugu in his voice** (IndicF5) | || 8 | Fit to the original timings | Telugu length ≠ English length || 9 | Remix over the original background | || 10 | QC + export | || 11 | *Optional*: fine-tune for a closer voice | Zero-shot gets timbre; fine-tuning gets delivery |**Before you distribute anything made here:** cloning a real person's voice needstheir consent, and the source recording is copyrighted. Label output as an AItranslation.

## 1 · SetupInstalls take ~3-4 minutes. Restart is not needed.

In [ ]:
#@title Install dependencies!pip -q install faster-whisper==1.1.0 demucs==4.0.1 soundfile librosa!pip -q install transformers accelerate sentencepiece anthropic!pip -q install git+https://github.com/ai4bharat/IndicF5.git 2>/dev/null || echo "IndicF5 loads via transformers trust_remote_code"!apt-get -qq install -y ffmpeg > /dev/nullimport torch, subprocessprint("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")print(subprocess.run(["ffmpeg","-version"],capture_output=True,text=True).stdout.splitlines()[0])

In [ ]:
#@title Get the pipeline code (align / prosody / QC / mixing)# This is the tested pipeline: isochrony fitting, Telugu syllable model,# ITU-R BT.1359 sync grading, reference-clip scoring.REPO_URL = "https://github.com/ManvithMadhuvarsu/JARVIS.git"  #@param {type:"string"}BRANCH   = "claude/sadhguru-telugu-lipsync-poc-38r9n3"        #@param {type:"string"}import os, subprocess, sysif not os.path.exists("/content/JARVIS"):    r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1",                        REPO_URL, "/content/JARVIS"],                       capture_output=True, text=True)    if r.returncode != 0:        print("Clone failed:", r.stderr.strip())        print("If the repo is private: upload the telugu_dub/ folder to /content "              "and set sys.path to /content/telugu_dub/src instead.")sys.path.insert(0, "/content/JARVIS/telugu_dub/src")from telugu_dub.align import fit_segments, sync_reportfrom telugu_dub.config import AlignCfg, AsrCfg, ProsodyCfgfrom telugu_dub.schema import Segmentfrom telugu_dub.telugu_prosody import (count_syllables, estimate_speech_duration,                                       syllable_budget, measure_rate)from telugu_dub.reference import find_candidatesfrom telugu_dub import qc as qc_modfrom telugu_dub.media import (time_stretch, assemble_timeline, write_pcm16,                              duration_of, extract_audio, ffmpeg)print("pipeline loaded")

## 2 · Upload your audioStart with **one file, 1-3 minutes**. Get it right, then scale up.

In [ ]:
#@title Uploadfrom google.colab import filesfrom pathlib import PathWORK = Path("/content/work"); WORK.mkdir(exist_ok=True)uploaded = files.upload()SRC = WORK / list(uploaded.keys())[0]SRC.write_bytes(list(uploaded.values())[0])print(SRC, f"{duration_of(SRC):.1f}s")# 16 kHz mono for analysis; keep the original for the final mixANALYSIS = WORK / "analysis_16k.wav"extract_audio(SRC, ANALYSIS, sr=16000)

## 3 · Separate the voice from the backgroundYour recording has music and ambience under the speech. If you dub over thewhole thing, the English voice stays audible underneath. Demucs splits it into`vocals` (what we replace) and `no_vocals` (what we keep).

In [ ]:
#@title Separate stemsSEPARATE = True  #@param {type:"boolean"}from pathlib import Pathif SEPARATE:    !python -m demucs --two-stems vocals -n htdemucs -o /content/stems "$SRC"    stem_dir = next(Path("/content/stems/htdemucs").iterdir())    VOCALS, BACKGROUND = stem_dir / "vocals.wav", stem_dir / "no_vocals.wav"    print("vocals:    ", VOCALS)    print("background:", BACKGROUND)else:    VOCALS, BACKGROUND = SRC, None# Transcribe the isolated vocals — cleaner input, noticeably fewer ASR errorsASR_INPUT = WORK / "vocals_16k.wav"extract_audio(VOCALS, ASR_INPUT, sr=16000)

## 4 · Transcribe`large-v3` on a T4 runs ~8-12× faster than real time.

In [ ]:
#@title Whisper large-v3from faster_whisper import WhisperModelMODEL_SIZE = "large-v3"  #@param ["large-v3", "medium.en", "small.en"]model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")segments_raw, info = model.transcribe(    str(ASR_INPUT), language="en", beam_size=5, word_timestamps=True,    vad_filter=True, vad_parameters={"min_silence_duration_ms": 300})chunks = [{"start": float(s.start), "end": float(s.end), "text": s.text.strip(),           "speaker": "SPEAKER_00"} for s in segments_raw]print(f"{len(chunks)} chunks, {sum(c['end']-c['start'] for c in chunks):.1f}s of speech")for c in chunks[:5]:    print(f"  [{c['start']:6.2f}-{c['end']:6.2f}] {c['text']}")

In [ ]:
#@title Build dubbing units (breath groups, not ASR chunks)from telugu_dub.config import AsrCfgfrom telugu_dub.stages.asr import build_unitsasr_cfg = AsrCfg(max_segment_seconds=8.0, min_segment_seconds=1.0,                 merge_gap_seconds=0.35)segments = build_units(chunks, asr_cfg)print(f"{len(segments)} dubbing units")for s in segments[:8]:    print(f"  {s.id:3d} [{s.start:6.2f}-{s.end:6.2f}] ({s.source_duration:4.1f}s) {s.text_src}")# READ THIS. Whisper mishears names and numbers. Fix them here — everything# downstream inherits these words.# segments[3].text_src = "corrected text"

## 5 · Translate like a dubbing translatorNot machine translation. The prompt below asks for meaning over structure,natural Telugu grammar, preserved tone, protected proper nouns — and a**syllable budget per line** derived from how long the speaker took to say it,because this audio has to fit back into the video.

In [ ]:
#@title Translation prompt (edit the style line for your content)DUBBING_SYSTEM_PROMPT = r'''You are a professional Telugu dubbing translator. Your output is spoken aloud bya voice actor and must fit the original speaker's timing.HOW TO TRANSLATE1. Translate the MEANING and INTENT, never the English sentence structure.   Rebuild the sentence the way a Telugu speaker would actually say it.2. Use natural spoken Telugu grammar and word order (verb-final, natural case   marking). If it reads like translated English, it is wrong.3. Preserve the tone exactly: sarcasm stays sarcastic, humour stays funny,   urgency stays urgent, warmth stays warm. A flat rendering of a joke is a   mistranslation.4. Choose the register by context — spoken discourse, not written prose. Use the   words a Telugu speaker uses at home, not textbook Telugu.5. DO NOT translate: proper names, place names, organisation names, product   names, technical terms, or established Sanskrit/yogic vocabulary that already   exists in Telugu. Keep English loanwords that Telugu speakers actually use   (e.g. "phone", "hospital", "office") — over-Sanskritising sounds foreign.6. LENGTH IS A HARD CONSTRAINT. Each line gives max_syllables. Going over means   the dub runs past the speaker's mouth. Cut English filler ("you know", "see",   "so", "I mean"), prefer shorter synonyms, and drop what carries no meaning.   Never pad a short line to fill the budget.7. Keep sentence-final punctuation — it becomes a pause in the dub.8. Telugu script only. No transliteration, no English gloss, no commentary.Return strict JSON: {"lines":[{"id":<int>,"te":"<telugu>","note":"<optional>"}]}'''STYLE = "Contemplative spiritual discourse: warm, plain-spoken, often wry. A village listener and a city listener must both follow it."  #@param {type:"string"}print(DUBBING_SYSTEM_PROMPT[:400], "...")

In [ ]:
#@title Translate with Claudeimport json, os, re, getpassimport anthropicos.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("ANTHROPIC_API_KEY: ")client = anthropic.Anthropic()MODEL = "claude-sonnet-5"  #@param {type:"string"}prosody = ProsodyCfg(target_syllables_per_second=6.5)   # recalibrated in step 7GLOSSARY = {"yoga":"యోగా","karma":"కర్మ","dharma":"ధర్మం","guru":"గురువు",            "meditation":"ధ్యానం","consciousness":"చైతన్యం","awareness":"అవగాహన",            "sadhana":"సాధన","mukti":"ముక్తి","Isha":"ఈశా","Sadhguru":"సద్గురు"}def translate_batch(batch, tighten=False):    payload = {"style": STYLE, "glossary": GLOSSARY,               "context_before": [s.text_src for s in segments                                  if s.id < batch[0].id and s.id >= batch[0].id-2],               "context_after": [s.text_src for s in segments                                 if s.id > batch[-1].id and s.id <= batch[-1].id+2],               "lines": [{"id": s.id, "en": s.text_src,                          "seconds": round(s.source_duration, 2),                          "max_syllables": syllable_budget(                              s.source_duration,                              prosody.target_syllables_per_second,                              0.85 if tighten else 0.95)} for s in batch]}    prefix = ("The previous attempt was TOO LONG. Cut every non-essential word "              "and stay under max_syllables.\n" if tighten else "")    msg = client.messages.create(        model=MODEL, max_tokens=8000, system=DUBBING_SYSTEM_PROMPT,        messages=[{"role":"user","content": prefix + json.dumps(payload, ensure_ascii=False, indent=1)}])    text = "".join(b.text for b in msg.content if b.type == "text")    data = json.loads(re.search(r"\{.*\}", text, re.S).group(0))    return {int(l["id"]): l["te"].strip() for l in data["lines"]}BATCH = 12for i in range(0, len(segments), BATCH):    batch = segments[i:i+BATCH]    out = translate_batch(batch)    for s in batch:        s.text_tgt = out.get(s.id, "")    # length-control retry: only for lines predicted to overrun    over = [s for s in batch            if estimate_speech_duration(s.text_tgt, prosody.target_syllables_per_second)            > s.source_duration * 1.12]    if over:        retry = translate_batch(over, tighten=True)        for s in over:            cand = retry.get(s.id, "")            if cand and count_syllables(cand) < count_syllables(s.text_tgt):                s.text_tgt = cand    print(f"  {i+len(batch)}/{len(segments)} lines", end="\r")print("\n")for s in segments[:10]:    est = estimate_speech_duration(s.text_tgt, prosody.target_syllables_per_second)    flag = "  <-- long" if est > s.source_duration*1.12 else ""    print(f"[{s.source_duration:4.1f}s slot | {est:4.1f}s est]{flag}\n  EN: {s.text_src}\n  TE: {s.text_tgt}\n")

In [ ]:
#@title Review and correct (do not skip)# This is where a Telugu speaker earns their keep. Print everything, fix what is# wrong, re-run this cell. Corrections survive the rest of the notebook.import jsonfor s in segments:    print(f"{s.id}\t{s.text_tgt}")# Example correction:# segments[7].text_tgt = "మీ ఇష్టం వచ్చినట్టు చేయండి."json.dump({"translations": {str(s.id): s.text_tgt for s in segments}},          open("/content/work/translations_te.json","w"), ensure_ascii=False, indent=1)print("\nsaved -> /content/work/translations_te.json")

## 6 · Pick the reference clip for voice cloningZero-shot cloning copies **whatever is in this clip** — timbre, room tone, mood,and any music behind it. So it is scored, not guessed: speech density, the 3-6 Hzsyllabic signature that separates voice from music, loudness consistency,clipping, and whether the cut points land in a pause.Use the **separated vocals**, not the original mix — background in the referencegets baked into every dubbed line.

In [ ]:
#@title Score and export candidate reference clipsimport shutilfrom pathlib import PathREF_SECONDS = 11  #@param {type:"slider", min:6, max:15, step:1}REF_DIR = Path("/content/work/ref"); REF_DIR.mkdir(parents=True, exist_ok=True)cands = find_candidates(str(ASR_INPUT), clip_seconds=REF_SECONDS, top_k=5)print(f"{'#':<3}{'start':<9}{'score':<8}{'speech':<9}{'syllabic':<10}{'steady'}")for i, c in enumerate(cands, 1):    dst = REF_DIR / f"ref_{i:02d}.wav"    ffmpeg(["-ss", str(c.start), "-t", str(REF_SECONDS), "-i", str(VOCALS),            "-ac","1","-ar","24000",            "-af", f"afade=t=in:d=0.02,afade=t=out:st={REF_SECONDS-0.02:.2f}:d=0.02,"                   "loudnorm=I=-20:TP=-2:LRA=7",            "-c:a","pcm_s16le", str(dst)])    print(f"{i:<3}{c.start:<9.1f}{c.score:<8.2f}{c.speech_fraction*100:<8.0f}%"          f" {c.modulation:<9.3f}{c.loudness_spread_db:.1f}")import IPython.display as ipdfor i in range(1, min(4, len(cands)+1)):    print(f"ref_{i:02d}"); ipd.display(ipd.Audio(str(REF_DIR/f"ref_{i:02d}.wav")))

In [ ]:
#@title Choose one and transcribe it EXACTLYCHOSEN = 1  #@param {type:"integer"}REF_AUDIO = str(REF_DIR / f"ref_{CHOSEN:02d}.wav")# IndicF5 aligns the generated speech against this transcript. Every word must# match, including false starts and filler. Auto-transcribe, then FIX BY EAR.ref_segs, _ = model.transcribe(REF_AUDIO, language="en", beam_size=5)REF_TEXT = " ".join(s.text.strip() for s in ref_segs).strip()print("Auto transcript:\n ", REF_TEXT)print("\n^ Listen to the clip above and correct this string if it is wrong.")# REF_TEXT = "the exact words spoken in the reference clip"

## 7 · Speak Telugu in his voice**IndicF5** (AI4Bharat) — the one open model that both speaks Telugu and clonesan arbitrary voice from a reference clip. Trained on 1,417 hours across 11 Indianlanguages.What zero-shot gives you: his **timbre**, close. What it does not give you: his**delivery** — the long pauses, the deliberate slowness, the timing of a joke.Step 11 (fine-tuning) is what closes that gap.

In [ ]:
#@title Load IndicF5 and synthesiseimport numpy as np, soundfile as sf, torchfrom transformers import AutoModelfrom pathlib import Pathtts = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)tts = tts.to("cuda" if torch.cuda.is_available() else "cpu")TTS_DIR = Path("/content/work/tts"); TTS_DIR.mkdir(parents=True, exist_ok=True)SR = 24000for s in segments:    if not s.text_tgt.strip():        continue    audio = tts(s.text_tgt, ref_audio_path=REF_AUDIO, ref_text=REF_TEXT)    audio = np.asarray(audio, dtype=np.float32)    if np.max(np.abs(audio)) > 1.0:          # model may return int16 range        audio = audio / 32768.0    dst = TTS_DIR / f"seg_{s.id:04d}.wav"    sf.write(str(dst), audio, SR)    s.tts_path, s.tts_duration = str(dst), len(audio) / SR    print(f"  {s.id+1}/{len(segments)}", end="\r")synth = sum(s.tts_duration or 0 for s in segments)src   = sum(s.source_duration for s in segments)print(f"\nTelugu {synth:.1f}s vs English {src:.1f}s  (ratio {synth/src:.2f})")

In [ ]:
#@title Recalibrate the speaking rate for THIS voice# Every timing decision derives from how fast this cloned voice actually speaks.rate = measure_rate([(s.text_tgt, s.tts_duration) for s in segments if s.tts_duration])print(rate)prosody = ProsodyCfg(target_syllables_per_second=rate["rate"])print(f"\nUsing {rate['rate']} syllables/s. If this differs a lot from 6.5, "      f"re-run the translation cell — the budgets were computed with the old value.")

## 8 · Fit it to the original timingsThe stage that decides whether this sounds dubbed or broken.Rules, in order: say it at natural tempo inside the original slot → bend thetempo within a band nobody can hear (0.90-1.15×) → borrow from the pause thatfollows → compress hard and flag it → let it overflow, bounded so one bad linecannot drag the rest out of sync.It never starts a line **before** the original onset: viewers detect audioleading picture at 45 ms but tolerate 125 ms of lag (ITU-R BT.1359-1).

In [ ]:
#@title Fitalign_cfg = AlignCfg(max_speedup=1.15, max_slowdown=0.90, hard_max_speedup=1.30,                     borrow_gap_fraction=0.80, max_shift=0.30)stats = fit_segments(segments, align_cfg, duration_of(SRC))print(stats.as_dict())result = qc_mod.evaluate(segments, max_speed=align_cfg.max_speedup)print("\n" + qc_mod.format_summary(result))for r in sync_report(segments):    if r["notes"] or abs(r["onset_drift"]) > 0.125:        print(f"  seg {r['id']}: drift {r['onset_drift']:+.2f}s speed {r['speed']} {r['notes']}")

**If QC fails**, fix the flagged lines in step 5 (shorter Telugu) and re-run 7-8. Do not push on with a failing gate.

## 9 · Render and remix over the original background

In [ ]:
#@title Build the dubbed trackfrom pathlib import PathSTRETCH = Path("/content/work/stretched"); STRETCH.mkdir(parents=True, exist_ok=True)clips = []for s in segments:    if not s.tts_path or s.fit_start is None:        continue    dst = STRETCH / f"seg_{s.id:04d}.wav"    if abs(s.speed - 1.0) < 1e-3:        ffmpeg(["-i", s.tts_path, "-ac","1","-ar",str(SR),"-c:a","pcm_s16le",str(dst)])    else:        time_stretch(s.tts_path, dst, s.speed, SR)   # pitch-preserving    clips.append((s.fit_start, str(dst)))DUB = "/content/work/dub_track.wav"write_pcm16(DUB, assemble_timeline(clips, duration_of(SRC), SR), SR)print("dub track:", duration_of(DUB), "s")

In [ ]:
#@title Mix: Telugu voice + original backgroundFINAL = "/content/work/final_te.mp3"BG_GAIN_DB = -3  #@param {type:"slider", min:-24, max:0, step:1}if BACKGROUND is not None:    # The English voice is gone (Demucs removed it), so the bed can sit high.    ffmpeg(["-i", str(BACKGROUND), "-i", DUB, "-filter_complex",            f"[0:a]aresample={SR},volume={BG_GAIN_DB}dB[bg];"            f"[1:a]aresample={SR}[dub];"            f"[bg][dub]amix=inputs=2:duration=longest:dropout_transition=0,"            f"loudnorm=I=-16:TP=-1.5:LRA=11[out]",            "-map","[out]","-c:a","libmp3lame","-b:a","192k", FINAL])else:    # No separation: duck the original under the dub instead.    ffmpeg(["-i", str(SRC), "-i", DUB, "-filter_complex",            f"[0:a]aresample={SR},volume=-8dB[bg];"            f"[1:a]aresample={SR},asplit=2[dub][key];"            f"[bg][key]sidechaincompress=threshold=0.03:ratio=12:attack=15:release=350[ducked];"            f"[ducked][dub]amix=inputs=2:duration=longest,loudnorm=I=-16:TP=-1.5:LRA=11[out]",            "-map","[out]","-c:a","libmp3lame","-b:a","192k", FINAL])import IPython.display as ipdprint("FINAL:"); ipd.display(ipd.Audio(FINAL))print("Telugu voice only:"); ipd.display(ipd.Audio(DUB))

## 10 · Subtitles, report, download

In [ ]:
#@title Export everythingimport jsonfrom telugu_dub.stages.mux import write_subtitlesSRT = "/content/work/dub_te.srt"write_subtitles(segments, SRT)json.dump({"qc": result.as_dict(), "align": stats.as_dict(),           "segments": [{"id":s.id,"src":[s.start,s.end],                         "dub":[s.fit_start, s.fit_duration],"speed":s.speed,                         "en":s.text_src,"te":s.text_tgt} for s in segments]},          open("/content/work/report.json","w"), ensure_ascii=False, indent=1)from google.colab import filesfor f in (FINAL, DUB, SRT, "/content/work/report.json",          "/content/work/translations_te.json"):    files.download(f)

## 11 · Optional: fine-tune for a closer voiceZero-shot gives timbre. Fine-tuning gives **delivery** — the pauses, the pacing,the emphasis that actually make someone sound like themselves.### What you need| | ||---|---|| Audio | **1-3 hours** of clean single-speaker speech. 30 min is the floor and it shows. || Quality | 24 kHz, no music, no audience, no overlapping speech. Run Demucs over everything first. || Transcripts | Every clip, accurate. Whisper large-v3 then a human pass. || Language | Telugu audio for a Telugu-speaking clone. English-only source data gives you an accent, not a voice. **This is the hard constraint** — if he has no Telugu recordings, fine-tune on his English and accept cross-lingual transfer, or stay with zero-shot. || GPU | A100 40 GB, ~8-20 h. A T4 will not finish in a Colab session. |### Data preparation

In [ ]:
#@title Build a fine-tuning dataset from a folder of recordings#@markdown Segments long recordings into 3-15 s single-speaker clips with#@markdown transcripts, in the layout F5-TTS expects.RAW_DIR = "/content/raw_audio"  #@param {type:"string"}OUT_DIR = "/content/dataset"    #@param {type:"string"}import os, json, csvfrom pathlib import Pathimport soundfile as sf, librosaPath(OUT_DIR, "wavs").mkdir(parents=True, exist_ok=True)rows, total = [], 0.0for src in sorted(Path(RAW_DIR).glob("*.*")):    if src.suffix.lower() not in {".mp3",".wav",".m4a",".flac"}:        continue    !python -m demucs --two-stems vocals -n htdemucs -o /content/_stems "$src" > /dev/null 2>&1    voc = next(Path("/content/_stems/htdemucs").rglob("vocals.wav"))    tmp = f"/content/_v_{src.stem}.wav"    extract_audio(voc, tmp, sr=16000)    segs, _ = model.transcribe(tmp, language="en", beam_size=5, vad_filter=True,                               vad_parameters={"min_silence_duration_ms":400})    audio24, _ = librosa.load(str(voc), sr=24000, mono=True)    for i, s in enumerate(segs):        dur = s.end - s.start        if not (3.0 <= dur <= 15.0):        # too short = no prosody, too long = OOM            continue        text = s.text.strip()        if len(text) < 10:            continue        clip = audio24[int(s.start*24000):int(s.end*24000)]        name = f"{src.stem}_{i:04d}.wav"        sf.write(f"{OUT_DIR}/wavs/{name}", clip, 24000)        rows.append((f"wavs/{name}", text)); total += durwith open(f"{OUT_DIR}/metadata.csv","w",newline="",encoding="utf-8") as f:    w = csv.writer(f, delimiter="|")    w.writerows(rows)print(f"{len(rows)} clips, {total/3600:.2f} hours -> {OUT_DIR}")print("Under 1 hour? Collect more before spending GPU time.")

### Fine-tuning runIndicF5 is F5-TTS fine-tuned for Indic languages, so F5-TTS's own trainer is thetool — start from the IndicF5 checkpoint rather than base F5-TTS, or you throwaway the Telugu phonetics.```bashgit clone https://github.com/SWivid/F5-TTS && cd F5-TTSpip install -e .# 1. prepare in F5-TTS's format (expects wavs/ + metadata.csv, as built above)python src/f5_tts/train/datasets/prepare_csv_wavs.py \    /content/dataset /content/dataset_f5# 2. fine-tune from the IndicF5 checkpointaccelerate launch src/f5_tts/train/finetune_cli.py \    --exp_name F5TTS_Base \    --dataset_name sadhguru_te \    --pretrain /content/indicf5_checkpoint.pt \    --learning_rate 1e-5 \    --batch_size_per_gpu 3200 --batch_size_type frame \    --max_samples 64 --epochs 40 --num_warmup_updates 2000 \    --save_per_updates 2000 --last_per_updates 500 \    --finetune True```**Watch for:** loss plateauing after ~10 epochs on small data (stop, you areoverfitting); the voice sounding right but the Telugu degrading (learning ratetoo high — drop to 5e-6); OOM (lower `batch_size_per_gpu`).**Evaluate honestly.** Hold out 10 clips the model never saw, synthesise the samesentences, and have a Telugu speaker rank fine-tuned vs zero-shot blind. If theycannot tell, you spent the GPU hours for nothing — go back to improving thetranslation instead, which is where the perceived quality actually lives.